<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-12</br>
</div>

</br>

# 학습 내용
>이번 장에서는 <strong>PyTorch 텐서와 Autograd(Tensor & Automatic Differentiation)</strong>에 대해 학습합니다.</br></br>
>텐서 생성, 연산, 자동 미분 메커니즘을 학습해봅시다.

</br>

# PyTorch 텐서 (Tensor)
> 텐서는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">다차원 배열</mark>로, NumPy의 ndarray와 유사하지만 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">GPU 연산과 자동 미분</mark>을 지원합니다.

><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">NumPy</mark>는 CPU 기반 수치 연산에 편리하지만, 딥러닝에는 두 가지 근본적 한계가 있습니다.</br>
> 첫째, GPU 연산이 불가능하여 수백만 파라미터의 행렬 곱을 CPU만으로 처리하면 수 시간~수 일이 걸리는 반면, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">GPU 병렬 연산은 수 분~수십 분</mark>에 처리합니다.</br>
> 둘째, 자동 미분(Autograd)이 없어 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">역전파(Backpropagation)</mark>에 필요한 수백만 편미분을 수동으로 구해야 합니다.</br></br>
> PyTorch는 `.to("cuda")` 한 줄로 GPU 전환이 가능하고, `backward()` 호출 한 번으로 모든 그래디언트를 자동 계산합니다. 즉, PyTorch = NumPy의 편의성 + GPU 가속 + 자동 미분입니다.

> 실제 신경망에서 파라미터 수는 수백만~수십억 개에 달하므로, 각 파라미터에 대해 손으로 미분식을 유도하고 코드로 구현하는 것은 현실적으로 불가능합니다.</br>
> 레이어 추가 시 미분식 재유도, 계산 실수로 인한 디버깅 난이도, 모델 구조 변경 시 미분 코드 전체 수정 등의 문제가 있으며, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Autograd는 이 모든 과정을 연산 그래프 추적으로 자동화</mark>하여 모델 설계에만 집중할 수 있게 합니다.

In [1]:
# TODO 1: torch를 import 해봅시다.

import torch

In [2]:
# TODO 2:
#  - x에 [1.0, 2.0, 3.0] 텐서를 저장해봅시다.
#  - matrix에 3x4 크기의 0으로 채워진 텐서를 저장해봅시다.
#  - random_tensor에 2x3 크기의 정규분포 난수 텐서를 저장해봅시다.

x = torch.tensor([1.0, 2.0, 3.0])
matrix = torch.zeros(3, 4)
random_tensor = torch.randn(2, 3)

print(f"x: {x}")
print(f"zeros:\n{matrix}")
print(f"randn:\n{random_tensor}")

x: tensor([1., 2., 3.])
zeros:
tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])
randn:
tensor([[-0.6887, -1.1446,  0.1373],
        [ 1.0081,  0.7937, -0.0621]])


</br>

## NumPy와 변환

In [3]:
# TODO 3: numpy 배열 [1, 2, 3]을 생성한 뒤, 텐서로 변환하고, 다시 NumPy 배열로 변환하여 각각 출력해봅시다.

import numpy as np

numpy_array = np.array([1, 2, 3])
tensor = torch.from_numpy(numpy_array)
numpy_back = tensor.numpy()

print(f"NumPy → Tensor: {tensor}")
print(f"Tensor → NumPy: {numpy_back}")
print(f"타입: {type(tensor)}, {type(numpy_back)}")

NumPy → Tensor: tensor([1, 2, 3])
Tensor → NumPy: [1 2 3]
타입: <class 'torch.Tensor'>, <class 'numpy.ndarray'>


💡메모리 공유
> `from_numpy`와 `numpy()`는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">메모리를 공유</mark>합니다.</br>
> 한쪽을 수정하면 다른 쪽도 변경됩니다. 독립 복사가 필요하면 `.clone()`을 사용하세요.

</br>

## 텐서 속성

In [4]:
# TODO 4: 3x4 크기의 정규분포 난수 텐서를 생성하고, shape, dtype, device 속성을 각각 출력해봅시다.

tensor = torch.randn(3, 4)
print(f"shape: {tensor.shape}")
print(f"dtype: {tensor.dtype}")
print(f"device: {tensor.device}")

shape: torch.Size([3, 4])
dtype: torch.float32
device: cpu


</br>

# Autograd (자동 미분)
> PyTorch의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">자동 미분 엔진</mark>으로, 연산 그래프를 추적하여 그래디언트를 자동 계산합니다.

In [5]:
# TODO 5:
#  - x에 [2.0, 3.0] 텐서를 기울기 추적 가능하게 저장해봅시다.
#  - y = x의 제곱 + 3x 식을 선언해봅시다.
#  - 역전파하여 기울기를 출력해봅시다.

x = torch.tensor([2.0, 3.0], requires_grad=True)
y = x ** 2 + 3 * x   # y = x² + 3x

z = y.sum()
z.backward()          # 역전파

# dy/dx = 2x + 3
print(f"x = {x.data}")
print(f"z = {z.item():.1f}")
print(f"x.grad = {x.grad}")   # [2*2+3, 2*3+3] = [7, 9]

x = tensor([2., 3.])
z = 28.0
x.grad = tensor([7., 9.])


</br>

## 계산 그래프 (Computational Graph)

### 역전파 체인룰 과정

> 역전파는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">출력에서 입력 방향</mark>으로 그래디언트를 전파합니다.</br>
> 각 노드에서 **상류(upstream) 그래디언트 × 로컬 그래디언트**를 곱해 나가는 것이 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">체인룰(Chain Rule)</mark>입니다.

<table style="width:100%; border-collapse:collapse; margin:8px 0;">
<tr style="background:#f5f5f5">
  <th style="text-align:center; padding:6px; border:1px solid #ddd;">단계</th>
  <th style="text-align:center; padding:6px; border:1px solid #ddd;">연산</th>
  <th style="text-align:center; padding:6px; border:1px solid #ddd;">로컬 그래디언트</th>
  <th style="text-align:center; padding:6px; border:1px solid #ddd;">체인룰 적용</th>
</tr>
<tr>
  <td style="text-align:center; padding:6px; border:1px solid #ddd;"><b>Step 1</b></td>
  <td style="text-align:center; padding:6px; border:1px solid #ddd;">z = sum(y)</td>
  <td style="text-align:center; padding:6px; border:1px solid #ddd;">∂z/∂z = 1 (출발점)</td>
  <td style="text-align:center; padding:6px; border:1px solid #ddd;"><b>1</b></td>
</tr>
<tr>
  <td style="text-align:center; padding:6px; border:1px solid #ddd;"><b>Step 2</b></td>
  <td style="text-align:center; padding:6px; border:1px solid #ddd;">z = y₁ + y₂</td>
  <td style="text-align:center; padding:6px; border:1px solid #ddd;">∂z/∂y = [1, 1]</td>
  <td style="text-align:center; padding:6px; border:1px solid #ddd;">1 × [1, 1] = <b>[1, 1]</b></td>
</tr>
<tr>
  <td style="text-align:center; padding:6px; border:1px solid #ddd;"><b>Step 3</b></td>
  <td style="text-align:center; padding:6px; border:1px solid #ddd;">y = x² + 3x</td>
  <td style="text-align:center; padding:6px; border:1px solid #ddd;">∂y/∂x = 2x + 3</td>
  <td style="text-align:center; padding:6px; border:1px solid #ddd;">[1, 1] × (2x + 3) = <b>2x + 3</b></td>
</tr>
<tr style="background:#FFEBEE">
  <td style="text-align:center; padding:6px; border:1px solid #ddd;"><b>결과</b></td>
  <td style="text-align:center; padding:6px; border:1px solid #ddd;">x = [2.0, 3.0]</td>
  <td style="text-align:center; padding:6px; border:1px solid #ddd;">대입</td>
  <td style="text-align:center; padding:6px; border:1px solid #ddd;"><b>x.grad = [7, 9]</b></td>
</tr>
</table>

<div style="text-align:center">

</div>

💡그래디언트 초기화
> `backward()`를 호출하면 그래디언트가 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">누적</mark>됩니다.</br>
> 학습 루프에서는 반드시 `optimizer.zero_grad()` 또는 `x.grad.zero_()`로 초기화해야 합니다.

</br>

## 그래디언트 비활성화

In [8]:
# TODO 6: torch.no_grad()를 사용하여 새 입력에 대해 추론을 수행해봅시다.
#  - weight에 [2.0, 3.0] 텐서를 기울기 추적 가능하게 저장해봅시다.
#  - bias에 [0.5] 텐서를 기울기 추적 가능하게 저장해봅시다.
#  - torch.no_grad() 안에서 new_input과 weight의 내적 + bias를 계산하여 prediction에 저장해봅시다.

weight = torch.tensor([2.0, 3.0], requires_grad=True)
bias = torch.tensor([0.5], requires_grad=True)
new_input = torch.tensor([1.0, 4.0])

with torch.no_grad():
    prediction = (new_input * weight).sum() + bias

print(f"추론 결과: {prediction.item():.1f}")
print(f"기울기 추적 여부: {prediction.requires_grad}")

추론 결과: 14.5
기울기 추적 여부: False


💡no_grad()의 효과
> 메모리 절약과 연산 속도 향상 효과가 있습니다.</br>
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">추론(Inference)과 평가(Evaluation)</mark> 시에는 항상 사용하세요.